# Carbon Mapper partner drop — products from a CSV

The partner scenario: a **CSV of pre-embargo plume records** lands (no
API discovery involved) and you want every product for its rows. Same
flow as [`products_reference.ipynb`](products_reference.ipynb), driven
entirely by the drop:

1. the CSV parses row-by-row into `CMRawPlume`;
2. the row's own URL columns give you a first product set directly;
3. the derived bundle + L2B tile give you everything else — versions
   resolved per row (drops routinely mix processing eras).

**Partner CSV vs public API record** — worth knowing:

| | partner CSV | public record |
|---|---|---|
| `rgb_tif` (geo) | ✅ column | ❌ (png quicklook only) |
| `wind_source_auto` | ✅ (e.g. `ecmwf_ifs`) | ❌ not exposed |
| sector | verbose `Oil & Gas (1B2)` | code `1B2` |
| asset URLs | api-gateway (stable, **Bearer-gated**) | signed CDN (expire) |
| timing | pre-embargo | ~30-day embargo observed |


In [ ]:
import csv
import os
from pathlib import Path

from georeader.readers.carbonmapper import (
    CarbonMapperConfig,
    CMPlumeImage,
    CMProductNotSelected,
    CMRawPlume,
    api_queries,
)
from georeader.readers.carbonmapper import products as P

# 429-resilient HTTP (Carbon Mapper rate-limits per account).
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

_session = requests.Session()
_session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=8, backoff_factor=2.0,
    status_forcelist=(429, 500, 502, 503, 504),
    respect_retry_after_header=True,
    allowed_methods=frozenset(["GET", "POST"]),
)))
requests.get, requests.post, requests.request = (
    _session.get, _session.post, _session.request,
)

# The partner drop: a CSV of pre-embargo plume records. Point
# CM_PARTNER_CSV at yours; the asset URLs inside are Bearer-gated
# api-gateway links, so a token is still needed to download products.
CSV_PATH = Path(os.environ.get("CM_PARTNER_CSV", "sample_plumes.csv"))
assert CSV_PATH.exists(), f"partner CSV not found: {CSV_PATH} — set CM_PARTNER_CSV"

TOKEN = CarbonMapperConfig.load().refresh_access_token()
os.environ["GDAL_HTTP_HEADERS"] = f"Authorization: Bearer {TOKEN}"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.TIF"
os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "3"
print(f"drop: {CSV_PATH.name}")


## 1 · Load the drop — CSV rows → typed plumes

One `CMRawPlume` per row. Notes: the export starts with a BOM
(`utf-8-sig` handles it), and the verbose `ipcc_sector` does not map
onto the `sector` code field — extract the code from the parentheses
when you need it.


In [ ]:
import re

import pandas as pd

# utf-8-sig strips the BOM the export carries on its first column.
rows = list(csv.DictReader(open(CSV_PATH, encoding="utf-8-sig")))
plumes = [CMRawPlume(**{k: (v or None) for k, v in r.items()}) for r in rows]


def sector_code(verbose):
    """Partner CSVs carry the verbose sector — the code sits in parens:
    'Oil & Gas (1B2)' -> '1B2'."""
    m = re.search(r"\(([^)]+)\)", verbose or "")
    return m.group(1) if m else verbose


summary = pd.DataFrame({
    "plume_id": [p.plume_id for p in plumes],
    "datetime": [r["datetime"][:10] for r in rows],
    "kg/h": [round(p.emission_auto) if p.emission_auto else None for p in plumes],
    "version": [p.version for p in plumes],
    "sector": [sector_code(r["ipcc_sector"]) for r in rows],
    "wind_source": [r["wind_source_auto"] for r in rows],
})
print(f"{len(plumes)} plumes, versions {sorted(summary.version.unique())}")
summary.sort_values("kg/h", ascending=False)


## 2 · Straight off the row — the CSV's own URL columns

Five URL columns per row. Unlike the public record, `rgb_tif` is a
real georeferenced product here — and the URLs are stable gateway
links (no signature expiry) but require the Bearer token.


In [ ]:
# Protagonist: the strongest plume in the drop. Its row's URL columns
# are everything the CSV links directly — note the partner CSV has a
# georeferenced rgb_tif the public API record does NOT.
plume = max(plumes, key=lambda p: p.emission_auto or 0)
PLUME_ID = plume.plume_id
print(f"{PLUME_ID}  {plume.emission_auto:.0f} kg/h  spec={plume.collection_spec}")
print()

url_fields = [
    ("plume_tif", plume.plume_tif, "geo", "l3a-vis plume.tif — RGBA vis; band-4 alpha = mask"),
    ("con_tif",   plume.con_tif,   "geo", "l3a-ime ime-cmf-concentrations.tif"),
    ("rgb_tif",   plume.rgb_tif,   "geo", "l3a-vis rgb.tif — PARTNER-ONLY as a record field"),
    ("plume_png", plume.plume_png, "png", "quicklook"),
    ("rgb_png",   plume.rgb_png,   "png", "quicklook"),
]
print(f"{'field':10s} {'set?':4s} {'geo?':4s} what it points at")
print("-" * 76)
for name, url, kind, desc in url_fields:
    print(f"{name:10s} {'yes' if url else '—':4s} {kind:4s} {desc}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyproj import Transformer
from shapely.ops import transform as shp_transform


def plot_outline(ax, geom_4326, dst_crs):
    """Reproject the plume outline into dst_crs and stroke it in red."""
    if geom_4326 is None:
        return
    tr = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    g = shp_transform(tr.transform, geom_4326)
    polys = list(g.geoms) if g.geom_type == "MultiPolygon" else [g]
    for p in polys:
        x, y = p.exterior.xy
        ax.plot(x, y, color="red", lw=1.4, solid_capstyle="round")


def to_display(reader, rgb=False):
    """Load a reader -> (array, extent, crs) ready for imshow."""
    geo = reader.load()
    arr = np.asarray(geo.values)
    b = geo.bounds
    extent = (b[0], b[2], b[1], b[3])
    if rgb:
        im = np.moveaxis(arr[:3], 0, -1).astype("float32")
        lo, hi = np.nanpercentile(im, [2, 98])
        return np.clip((im - lo) / max(hi - lo, 1e-9), 0, 1), extent, str(geo.crs)
    a = arr.squeeze().astype("float32")
    nd = reader.nodata
    if nd is not None:
        a = np.where(a == nd, np.nan, a)
    return a, extent, str(geo.crs)


In [ ]:
# Download the three georeferenced record products into a local "drop"
# directory — the archive/EXPAND step of a partner pipeline — and plot
# them. Gateway URLs need the Bearer token.
DROP = Path("/tmp/cm_notebooks") / f"drop_{PLUME_ID}"
DROP.mkdir(parents=True, exist_ok=True)

drop_files = {}
for key, url in [("plume.tif", plume.plume_tif),
                 ("ime-cmf-concentrations.tif", plume.con_tif),
                 ("rgb.tif", plume.rgb_tif)]:
    dest = DROP / key
    if not dest.exists():
        r = requests.get(url, headers={"Authorization": f"Bearer {TOKEN}"},
                         timeout=120)
        r.raise_for_status()
        dest.write_bytes(r.content)
    drop_files[key] = str(dest)
    print(f"{key:28s} {dest.stat().st_size // 1024:>5d} KB")

import rasterio

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)
for ax, (key, cmap) in zip(axes, [("plume.tif", None),
                                  ("ime-cmf-concentrations.tif", "plasma"),
                                  ("rgb.tif", None)]):
    with rasterio.open(drop_files[key]) as ds:
        arr, b, nd = ds.read(), ds.bounds, ds.nodata
    if arr.shape[0] >= 3:
        ax.imshow(np.moveaxis(arr, 0, -1),
                  extent=(b.left, b.right, b.bottom, b.top), origin="upper")
    else:
        a = arr.squeeze().astype("float32")
        if nd is not None:
            a = np.where(a == nd, np.nan, a)
        im = ax.imshow(a, extent=(b.left, b.right, b.bottom, b.top),
                       origin="upper", cmap=cmap)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="ppm·m")
    ax.set_title(f"{key} — {arr.shape[-1]}×{arr.shape[-2]}px", fontsize=9)
    ax.set_aspect("equal")
fig.suptitle(f"Products linked directly from the CSV row · {PLUME_ID}", fontsize=11)
plt.show()


## 3 · The full product set, derived from the row

Everything the CSV does *not* link — `plume-concentrations.tif`, both
outlines, `ime-cmf-mask.tif` — via the same `CMPlumeImage` derivation
the API path uses.


In [ ]:
# The CSV row is a full CMRawPlume, so the derived bundle works exactly
# as with API records — the collection spec is parsed from the row's
# own plume_tif URL (this drop mixes v3, v3c and v3d rows; each plume
# resolves its own version).
img = CMPlumeImage.from_cmrawplume(plume, token=TOKEN)
print(img)


In [ ]:
import time

# Every L3A raster, with the canonical outline overlaid. All of these
# are thumbnail-grade by design — analysis-grade rasters come from the
# L2B parent in the next section.
alpha = img.load_alpha_mask()   # plume.tif band-4 alpha -> boolean mask

panels = [
    ("mask (plume.tif alpha)", None, "Greys", "binary plume mask"),
    ("concentrations", img.concentrations, "plasma", "full plume window"),
    ("ime_concentrations", img.ime_concentrations, "plasma",
     "IME-clipped (emission_auto integrand)"),
    ("ime_mask", img.ime_mask, "Greys", "pixels integrated for emission_auto"),
    ("rgb", img.rgb, None, "true-colour context"),
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8.6), constrained_layout=True)
axes = axes.ravel()
axes[-1].set_axis_off()

for ax, (name, reader, cmap, note) in zip(axes, panels):
    if name.startswith("mask"):
        if alpha is None:
            ax.set_axis_off(); ax.set_title("mask (absent)"); continue
        b = alpha.bounds
        arr = np.asarray(alpha.values).squeeze()
        ax.imshow(arr, extent=(b[0], b[2], b[1], b[3]), origin="upper",
                  cmap="Greys", interpolation="nearest")
        crs = str(alpha.crs)
    elif reader is None:
        ax.set_axis_off(); ax.set_title(f"{name} (absent)"); continue
    else:
        time.sleep(1)  # be gentle with the asset gateway
        arr, extent, crs = to_display(reader, rgb=(cmap is None))
        if cmap is None:
            ax.imshow(arr, extent=extent, origin="upper")
        else:
            im = ax.imshow(arr, extent=extent, origin="upper", cmap=cmap)
            if "concentrations" in name:
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="ppm·m")
    plot_outline(ax, img.outline, crs)
    h, w = arr.shape[:2]
    ax.set_title(f"{name} — {w}×{h}px\n{note}", fontsize=9)
    ax.set_aspect("equal")
fig.suptitle(f"L3A per-plume products · {PLUME_ID}", fontsize=11)
plt.show()


## 4 · The L2B parent tile

Same as the API path: whole-swath scene rasters keyed by the scene
name embedded in the row's `plume_id`.


In [ ]:
# The L2B parent resolves from the row's spec — the plume's own version
# is probed first, older candidates as backup. Lazy: nothing read yet.
ir = img.tile
print(ir)
print(f"\nL2B collection: {str(ir.asset_paths['cmf']).split('/')[-6]}")


In [ ]:
from pathlib import Path

# Cache the three bands we plot locally, so repeated reads don't hammer
# the gateway. Idempotent — re-runs use the files.
CACHE = Path("/tmp/cm_notebooks") / PLUME_ID
CACHE.mkdir(parents=True, exist_ok=True)
for band in ("cmf", "rgb", "uncertainty", "artifact-mask"):
    local = CACHE / f"{band}.tif"
    if not local.exists():
        r = requests.get(str(ir.asset_paths[band]),
                         headers={"Authorization": f"Bearer {TOKEN}"},
                         stream=True, timeout=120)
        if r.status_code == 404:
            # Optional asset — not every scene publishes every band
            # (e.g. artifact-mask is absent on some v3d scenes). Drop
            # the key so the reader's accessor returns None cleanly.
            ir.asset_paths.pop(band, None)
            print(f"{band:12s} not published for this scene (404)")
            continue
        r.raise_for_status()
        local.write_bytes(r.content)
    ir.asset_paths[band] = str(local)
    print(f"{band:12s} {local.stat().st_size // 1024:>6d} KB")


In [ ]:
# The FULL scene — these are the same whole-swath GeoTIFFs the STAC
# items point at, not crops. The red outline marks where the plume
# sits in the swath (analysis-grade crops of it come next).
full_panels = [
    ("cmf", "magma", "ppm·m"),
    ("rgb", None, None),
    ("uncertainty", "viridis", "ppm·m"),
    ("artifact_mask", "Greys", None),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 11), constrained_layout=True)
for ax, (band, cmap, label) in zip(axes.ravel(), full_panels):
    reader = getattr(ir, band)
    if reader is None:
        ax.set_axis_off(); ax.set_title(f"{band} (absent)"); continue
    arr, extent, crs = to_display(reader, rgb=(cmap is None))
    if cmap is None:
        ax.imshow(arr, extent=extent, origin="upper")
    else:
        im = ax.imshow(arr, extent=extent, origin="upper", cmap=cmap)
        if label:
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=label)
    plot_outline(ax, img.outline, crs)
    h, w = arr.shape[:2]
    ax.set_title(f"{band} — full scene, {w}×{h}px", fontsize=9)
    ax.set_aspect("equal")
fig.suptitle(f"Full L2B swath · {ir.scene_id}", fontsize=11)
plt.show()


In [ ]:
# The headline workflow: analysis-grade crops at L2B native resolution,
# next to the thumbnail-grade L3A product they replace.
crop_cmf = img.tile_cmf(pad_px=64)
crop_rgb = img.tile_rgb(pad_px=64)
crop_unc = img.tile_uncertainty(pad_px=64)

fig, axes = plt.subplots(2, 2, figsize=(11, 9.6), constrained_layout=True)
axes = axes.ravel()

arr, extent, crs = to_display(img.concentrations)
im = axes[0].imshow(arr, extent=extent, origin="upper", cmap="magma")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="ppm·m")
plot_outline(axes[0], img.outline, crs)
axes[0].set_title(f"L3A thumbnail — {arr.shape[1]}×{arr.shape[0]}px", fontsize=9)


def show_crop(ax, crop, title, cmap):
    a = np.asarray(crop.values)
    if cmap is None:
        im3 = np.moveaxis(a[:3], 0, -1).astype("float32")
        lo, hi = np.nanpercentile(im3, [2, 98])
        a = np.clip((im3 - lo) / max(hi - lo, 1e-9), 0, 1)
    else:
        a = np.where(a.squeeze() == -9999, np.nan, a.squeeze()).astype("float32")
    b = crop.bounds
    if cmap is None:
        ax.imshow(a, extent=(b[0], b[2], b[1], b[3]), origin="upper")
    else:
        im = ax.imshow(a, extent=(b[0], b[2], b[1], b[3]),
                       origin="upper", cmap=cmap)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="ppm·m")
    plot_outline(ax, img.outline, str(crop.crs))
    ax.set_title(f"{title} — {a.shape[1]}×{a.shape[0]}px @ L2B native", fontsize=9)


show_crop(axes[1], crop_cmf, "tile_cmf(pad_px=64)", "magma")
show_crop(axes[2], crop_rgb, "tile_rgb(pad_px=64)", None)
show_crop(axes[3], crop_unc, "tile_uncertainty(pad_px=64)", "viridis")
for ax in axes:
    ax.set_aspect("equal")
fig.suptitle(f"Thumbnail vs analysis-grade · {PLUME_ID}", fontsize=11)
plt.show()


## 5 · File-mode — the materialised drop

A partner pipeline archives the drop and ingests **from files**
(archive-then-ingest). The same bundle API works over local paths —
no token, no network.


In [ ]:
# File-mode: point the bundle at the MATERIALISED drop instead of URLs
# — the ingest side of archive-then-ingest. Select the products the
# drop contains plus PLUME_OUTLINE: the drop has no outline file, so
# the property transparently falls back to vectorising the band-4
# alpha of plume.tif (it logs a warning to flag the non-canonical path).
offline = CMPlumeImage(
    plume_id=PLUME_ID,
    urls=drop_files,  # {asset key -> local path}, from the download cell
    products=(P.PLUME_TIF, P.IME_CONCENTRATIONS, P.RGB_TIF, P.PLUME_OUTLINE),
)
print(f"offline mask:     {offline.mask.shape}")
print(f"offline IME:      {offline.ime_concentrations.shape}")
print(f"offline rgb:      {offline.rgb.shape}")
outline_offline = offline.outline   # alpha-fallback, no network
print(f"offline outline:  bounds={tuple(round(b, 4) for b in outline_offline.bounds)}")

# Unselected products still fail loudly — no silent Nones offline either.
try:
    offline.concentrations
except CMProductNotSelected as exc:
    print(f"offline.concentrations -> {type(exc).__name__}")


## Notes for pipeline builders

- **Persist files, not URLs** — gateway links are stable but
  Bearer-gated; the drop archive is your provenance.
- **Versions vary per row** (this sample mixes v3 / v3c / v3d) — always
  resolve the spec from each row, never assume one version per drop.
- **No outline in the drop** — the canonical `plume-outline.geojson` is
  derivable (§3) or the band-4 alpha fallback covers file-mode (§5).
- **Sources**: pre-embargo plumes are usually not yet DBSCAN-clustered;
  expect `get_source_for_plume` to return `None` until they go public.
